In [ ]:
import torch
import torch.nn as nn
import numpy as np
import pickle
from collections import defaultdict
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import normalize
from sklearn.decomposition import PCA
from sklearn.metrics.pairwise import cosine_similarity
from tqdm import tqdm
import gc
import warnings
warnings.filterwarnings('ignore')

In [ ]:
embedder_name = 'qwen35_4b_middle'
print("Загрузка эмбеддингов...")
with open('/content/drive/MyDrive/Master_thesis/qwen35_4b_middle_embeddings.pkl', 'rb') as f:
    all_embeddings_dict = pickle.load(f)

In [ ]:
valid_items = []
for key, data in all_embeddings_dict.items():
  if data['form'] != '#NULL' and data['sem_class'] != '_':
    valid_items.append({
        'key': key,
        'embedding': data['embedding'],
        'lemma': data.get('lemma', data['form']),
        'upos': data.get('upos')
    })

print(f"Отобрано {len(valid_items)} токенов")
if not valid_items: raise ValueError("Нет валидных токенов.")

In [ ]:
print("PCA...")
X_raw = np.stack([item['embedding'] for item in valid_items]).astype(np.float32)
pca = PCA(n_components=256, random_state=42)
X_pca = pca.fit_transform(X_raw)
X_pca = normalize(X_pca, norm='l2', axis=1)
del X_raw; gc.collect()

In [ ]:
print("Поиск контекстуально-одинаковых пар...")
lemma_to_indices = defaultdict(list)
for i, item in enumerate(valid_items):
    lemma_to_indices[item['lemma']].append(i)

anc_idx, pos_idx = [], []
SIM_THRESHOLD = 0.4
MAX_LEMMA_SAMPLES = 400

for lem, idxs in tqdm(lemma_to_indices.items(), desc="Группировка"):
    if len(idxs) < 2: continue

    if len(idxs) > MAX_LEMMA_SAMPLES:
        sample = np.random.choice(idxs, MAX_LEMMA_SAMPLES, replace=False)
    else:
        sample = idxs

    X_lem = X_pca[sample]
    sim_matrix = cosine_similarity(X_lem)
    np.fill_diagonal(sim_matrix, -1.0)  # исключаем само-сравнение

    rows, cols = np.where(sim_matrix > SIM_THRESHOLD)
    for r, c in zip(rows, cols):
        anc_idx.append(sample[r])
        pos_idx.append(sample[c])

# Убираем дубликаты пар (i,j) == (j,i)
seen = set()
pairs = []
for a, p in zip(anc_idx, pos_idx):
    key = (min(a, p), max(a, p))
    if key not in seen:
        seen.add(key)
        pairs.append((a, p))

anc_idx = np.array([p[0] for p in pairs], dtype=np.int64)
pos_idx = np.array([p[1] for p in pairs], dtype=np.int64)
print(f"Создано {len(anc_idx)} пар одного смысла (многозначные леммы разделены)")

if len(anc_idx) < 500:
    raise RuntimeError("Слишком мало пар")

In [ ]:
class ContrastiveProjector(nn.Module):
    def __init__(self, input_dim, hidden_dim=512, output_dim=256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(hidden_dim, output_dim)
        )
    def forward(self, x): return self.net(x)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
X_tensor = torch.FloatTensor(X_pca).to(device)
proj = ContrastiveProjector(X_pca.shape[1], 512, 256).to(device)
optimizer = torch.optim.AdamW(proj.parameters(), lr=1e-3)
temperature = 0.05

class PairDataset(Dataset):
    def __init__(self, anc, pos): self.anc, self.pos = anc, pos
    def __len__(self): return len(self.anc)
    def __getitem__(self, idx): return self.anc[idx], self.pos[idx]

loader = DataLoader(PairDataset(anc_idx, pos_idx), batch_size=256, shuffle=True, drop_last=True)

print("Запуск контрастивного обучения...")
proj.train()
for epoch in range(100):
    total_loss = 0
    for anc_i, pos_i in tqdm(loader, desc=f"Epoch {epoch+1}"):
        anc_i, pos_i = anc_i.to(device), pos_i.to(device)

        h1 = proj(X_tensor[anc_i])
        h2 = proj(X_tensor[pos_i])

        h1 = nn.functional.normalize(h1, dim=1)
        h2 = nn.functional.normalize(h2, dim=1)

        sim = torch.matmul(h1, h2.T) / temperature
        labels = torch.arange(h1.shape[0], device=device)
        loss = (nn.functional.cross_entropy(sim, labels) +
                nn.functional.cross_entropy(sim.T, labels)) / 2.0

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    print(f"Epoch {epoch+1}/{10} | Loss: {total_loss/len(loader):.4f}")

In [ ]:
proj.eval()
with torch.no_grad():
    improved = []
    bs = 4096
    for i in tqdm(range(0, len(X_tensor), bs), desc="Transform"):
        h = proj(X_tensor[i:i+bs])
        h = nn.functional.normalize(h, dim=1)
        improved.append(h.cpu().numpy())
    X_improved = np.vstack(improved)

print("Сохранение...")
for i, item in enumerate(valid_items):
    all_embeddings_dict[item['key']]['embedding'] = X_improved[i]

save_path = f'/content/drive/MyDrive/Master_thesis/{embedder_name}_simcse_refined3.pkl'
with open(save_path, 'wb') as f:
    pickle.dump(all_embeddings_dict, f)

In [ ]:
import numpy as np
import pickle
from sklearn.metrics import silhouette_score
from sklearn.cluster import KMeans

X = []
y_sc = []
for key, data in all_embeddings_dict.items():
  if data['form'] != '#NULL' and data['sem_class'] != '_':
    X.append(np.array(data['embedding']))
    y_sc.append(data['sem_class'])
X = np.stack(X)
y_sc = np.array(y_sc)

np.random.seed(42)
idx = np.random.choice(len(X), min(5000, len(X)), replace=False)
X_sample = X[idx]
y_sample = y_sc[idx]
mask = y_sample != '_'

if mask.sum() > 100:
    n_clusters = len(set(y_sample[mask]))
    labels = KMeans(n_clusters=n_clusters, random_state=42, n_init=5).fit_predict(X_sample[mask])

    sil = silhouette_score(X_sample[mask], labels)
    print(f"Silhouette после SimCSE: {sil:.3f}")
    print(f"Кластеров: {n_clusters}, Токенов в оценке: {mask.sum()}")